In [3]:
import openeo

In [4]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


In [5]:
aoi = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [
              112.6654991,
              -7.0171347
            ],
            [
              112.6912024,
              -7.1679391
            ],
            [
              112.7995564,
              -7.0918788
            ],
            [
              112.6993392,
              -7.0334062
            ],
            [
              112.6654991,
              -7.0171347
            ]
          ]
        ]
      }
    }
  ]
}

In [6]:
CO = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-24", "2026-08-24"],
    spatial_extent = {
    "west": 112.6654991,   # Longitude minimum
    "south": -7.1679391,   # Latitude minimum
    "east": 112.7995564,   # Longitude maksimum
    "north": -7.0171347    # Latitude maksimum
    },
    bands=["NO2"],
)

In [7]:
# Now aggregate by day to avoid having multiple data per day
CO = CO.aggregate_temporal_period(reducer="mean", period="day")

# let's create a spatial aggregation to generate mean timeseries data
CO = CO.aggregate_spatial(reducer="mean", geometries=aoi)

In [8]:
# Create a datacube for period after COVID lockdowns

s5post = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-24", "2026-08-24"],
    spatial_extent = {
    "west": 112.6654991,   # Longitude minimum
    "south": -7.1679391,   # Latitude minimum
    "east": 112.7995564,   # Longitude maksimum
    "north": -7.0171347    # Latitude maksimum
    },
    bands=["NO2"],
)

# Now aggregate by day to avoid having multiple data per day
s5post = s5post.aggregate_temporal_period(reducer="mean", period="day")

# Now create a spatial aggregation to generate mean timeseries data
s5post = s5post.aggregate_spatial(reducer="mean", geometries=aoi)

In [9]:
job = s5post.execute_batch(title="NO2 Bangkalan", outputfile="../data/nc/polutan_NO2_bangkalan.nc")

0:00:00 Job 'j-26090802595647c2adc2f82007410287': send 'start'
0:00:06 Job 'j-26090802595647c2adc2f82007410287': queued (progress 0%)
0:00:15 Job 'j-26090802595647c2adc2f82007410287': queued (progress 0%)
0:00:21 Job 'j-26090802595647c2adc2f82007410287': queued (progress 0%)
0:00:30 Job 'j-26090802595647c2adc2f82007410287': queued (progress 0%)
0:00:40 Job 'j-26090802595647c2adc2f82007410287': queued (progress 0%)
0:00:52 Job 'j-26090802595647c2adc2f82007410287': queued (progress 0%)
0:01:08 Job 'j-26090802595647c2adc2f82007410287': queued (progress 0%)
0:01:28 Job 'j-26090802595647c2adc2f82007410287': queued (progress 0%)
0:01:52 Job 'j-26090802595647c2adc2f82007410287': queued (progress 0%)
0:02:23 Job 'j-26090802595647c2adc2f82007410287': running (progress N/A)
0:03:00 Job 'j-26090802595647c2adc2f82007410287': running (progress N/A)
0:03:47 Job 'j-26090802595647c2adc2f82007410287': running (progress N/A)
0:04:46 Job 'j-26090802595647c2adc2f82007410287': running (progress N/A)
0:05:4

### Ekspor Data NetCDF (.nc) ke CSV

Sel di bawah ini digunakan untuk memproses file `.nc` hasil crawling di atas, mengekstrak data tanggal (`tanggal`) dan nilai konsentrasi polutan, lalu menyimpannya dalam format `.csv` ke folder `data/csv/`.

In [10]:
import os
import shutil
import tempfile
import xarray as xr
import pandas as pd

# Konfigurasi path file input (.nc) dan output (.csv)
nc_file_path = "../data/nc/polutan_NO2_bangkalan.nc"
csv_file_path = "../data/csv/polutan_no2_bangkalan.csv"

print(f"Membaca file: {nc_file_path}")

is_temp = False
try:
    ds = xr.open_dataset(nc_file_path, engine="netcdf4")
except Exception as e:
    print("Mendeteksi unicode path pada Windows, menggunakan fallback menyalin file ke folder temp...")
    temp_dir = tempfile.gettempdir()
    temp_nc_path = os.path.join(temp_dir, "temp_NO2_bangkalan.nc")
    shutil.copy2(nc_file_path, temp_nc_path)
    ds = xr.open_dataset(temp_nc_path, engine="netcdf4")
    is_temp = True

# 1. Konversi dataset NetCDF ke Pandas DataFrame
df_raw = ds.to_dataframe().reset_index()

# 2. Ambil data tanggal ('t') dan nilai polutan ('NO2')
df_raw['tanggal'] = pd.to_datetime(df_raw['t']).dt.strftime('%Y-%m-%d')
no2_data = dict(zip(df_raw['tanggal'], df_raw['NO2']))

# 3. Buat deret tanggal lengkap dari 2025-08-24 sampai 2026-08-24
full_dates = pd.date_range(start="2025-08-24", end="2026-08-24", freq="D")

# 4. Buat DataFrame baru dengan semua tanggal
df_result = pd.DataFrame({
    "tanggal": full_dates.strftime("%Y-%m-%d")
})

# 5. Petakan nilai NO2 ke tanggal yang sesuai
df_result["NO2"] = df_result["tanggal"].map(no2_data)

# 6. Simpan hasil ke file CSV
os.makedirs(os.path.dirname(csv_file_path), exist_ok=True)
df_result.to_csv(csv_file_path, index=False)

print(f"\nBerhasil menyimpan ke: {csv_file_path}")
print("Jumlah baris:", len(df_result))
print("\nContoh data:")
print(df_result.head(10))

ds.close()
if is_temp:
    try:
        os.remove(temp_nc_path)
    except Exception:
        pass


Membaca file: ../data/nc/polutan_NO2_bangkalan.nc
Mendeteksi unicode path pada Windows, menggunakan fallback menyalin file ke folder temp...

Berhasil menyimpan ke: ../data/csv/polutan_no2_bangkalan.csv
Jumlah baris: 366

Contoh data:
      tanggal       NO2
0  2025-08-24  0.000030
1  2025-08-25  0.000040
2  2025-08-26  0.000046
3  2025-08-27       NaN
4  2025-08-28  0.000029
5  2025-08-29  0.000029
6  2025-08-30  0.000006
7  2025-08-31  0.000019
8  2025-09-01  0.000033
9  2025-09-02  0.000014
